# Debiasing distribution shift

Measures how much each debiasing step changes the activation distribution at the debiased layer.
Analyzed separately for positive and negative examples of each concept.

**Two debiasing regimes:**
- **Single**: raw activations at layer L vs. orthogonally projected activations at layer L.
  (`data/activations/raw/` vs `data/activations/debiased/single/{concept}/{method}/from_layer_L/`)
- **Sequential**: activations at layer L before debiasing step L (pre) vs. after (post).
  (`data/activations/debiased/sequential/{concept}/{method}/layer_L_pre.parquet` vs `layer_L_post.parquet`)

**Metrics:**
- MMD² with RBF kernel (median bandwidth heuristic) — non-parametric distribution distance
- Delta in mean |skewness| and |excess kurtosis| before vs. after debiasing

Data: **test split** with subsampling (max 300 per class for MMD speed).

Output: `notebooks/results/distributions_analysis/debiasing_shift/`

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent

In [ ]:
import random, numpy as np, torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONCEPTS = [
    'eyeglasses', 'bald', 'chubby', 'wearing_hat', 'male']
METHODS     = ['diff_means', 'lr', 'pclarc']
NUM_LAYERS  = 24
SPLIT       = 'test'
MMD_SUBSAMPLE = 300   # max samples per class for MMD (speed vs accuracy tradeoff)
MMD_SEED      = 42

RAW_DIR    = ROOT / 'data' / 'activations' / 'raw' / SPLIT
DEB_DIR    = ROOT / 'data' / 'activations' / 'debiased'
METADATA   = ROOT / 'data' / 'metadata.csv'
OUT_DIR    = ROOT / 'notebooks' / 'results' / 'distributions_analysis' / 'debiasing_shift'
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert RAW_DIR.exists(),  f'Missing raw activations: {RAW_DIR}'
assert METADATA.exists(), f'Missing: {METADATA}'

df_meta  = pd.read_csv(METADATA)
df_split = df_meta[df_meta['split'] == SPLIT].reset_index(drop=True)

print(f'Split     : {SPLIT} ({len(df_split)} images)')
print(f'Concepts  : {CONCEPTS}')
print(f'Methods   : {METHODS}')
print(f'MMD sub   : {MMD_SUBSAMPLE} samples/class')

In [ ]:
rng = np.random.default_rng(MMD_SEED)


def subsample(X, n):
    if len(X) <= n:
        return X
    idx = rng.choice(len(X), n, replace=False)
    return X[idx]


def mmd_rbf(X, Y):
    """Unbiased MMD² with RBF kernel (median bandwidth heuristic)."""
    X = subsample(X.astype(np.float64), MMD_SUBSAMPLE)
    Y = subsample(Y.astype(np.float64), MMD_SUBSAMPLE)
    n, m = len(X), len(Y)
    if n < 4 or m < 4:
        return np.nan

    # Median heuristic on a subsample of pairwise distances
    XY = np.vstack([X, Y])
    n_med = min(300, len(XY))
    idx = rng.choice(len(XY), n_med, replace=False)
    sub = XY[idx]
    sq = np.sum((sub[:, None] - sub[None, :]) ** 2, axis=-1)
    median_sq = np.median(sq[sq > 0])
    if median_sq < 1e-10:
        return 0.0
    gamma = 1.0 / (2.0 * median_sq)

    def rbf(A, B):
        d2 = (np.sum(A**2, axis=1, keepdims=True)
              + np.sum(B**2, axis=1)
              - 2 * A @ B.T)
        return np.exp(-gamma * np.clip(d2, 0, None))

    Kxx = rbf(X, X)
    Kyy = rbf(Y, Y)
    Kxy = rbf(X, Y)
    mmd2 = ((Kxx.sum() - np.trace(Kxx)) / (n * (n - 1))
            + (Kyy.sum() - np.trace(Kyy)) / (m * (m - 1))
            - 2 * Kxy.mean())
    return float(max(0.0, mmd2))


def marginal_stats(X):
    """Returns mean |skewness| and mean |excess kurtosis| over dimensions."""
    if len(X) < 8:
        return np.nan, np.nan
    return float(np.abs(skew(X, axis=0)).mean()), float(np.abs(kurtosis(X, axis=0)).mean())


def load_raw(layer_idx, concept, df_meta_split):
    """Returns (X_pos, X_neg) from raw activations."""
    path = RAW_DIR / f'layer_{layer_idx:02d}.parquet'
    if not path.exists():
        return None, None
    df = pd.read_parquet(path).merge(
        df_meta_split[['filename', concept]], on='filename', how='inner'
    )
    feat_cols = [c for c in df.columns if c not in ('filename', concept)]
    X = df[feat_cols].values.astype(np.float32)
    y = df[concept].values
    return X[y == 1], X[y == 0]


def load_parquet_feats(path, concept, df_meta_split):
    """Load parquet, join labels, return (X_pos, X_neg)."""
    if not path.exists():
        return None, None
    df = pd.read_parquet(path).merge(
        df_meta_split[['filename', concept]], on='filename', how='inner'
    )
    feat_cols = [c for c in df.columns if c not in ('filename', concept)]
    X = df[feat_cols].values.astype(np.float32)
    y = df[concept].values
    return X[y == 1], X[y == 0]

In [ ]:
records_single = []

print('=== Single debiasing shift ===')
for concept in tqdm(CONCEPTS, desc='Concepts'):
    if concept not in df_split.columns:
        continue
    for method in METHODS:
        for layer_idx in range(NUM_LAYERS):
            post_path = (DEB_DIR / 'single' / concept / method
                         / f'from_layer_{layer_idx:02d}' / SPLIT
                         / f'layer_{layer_idx:02d}.parquet')

            X_raw_pos, X_raw_neg = load_raw(layer_idx, concept, df_split)
            X_deb_pos, X_deb_neg = load_parquet_feats(post_path, concept, df_split)

            if X_raw_pos is None or X_deb_pos is None:
                continue

            for label, X_raw, X_deb in [
                ('pos', X_raw_pos, X_deb_pos),
                ('neg', X_raw_neg, X_deb_neg),
            ]:
                if len(X_raw) < 4 or len(X_deb) < 4:
                    continue

                sk_raw,  ku_raw  = marginal_stats(X_raw)
                sk_deb,  ku_deb  = marginal_stats(X_deb)
                mmd = mmd_rbf(X_raw, X_deb)

                records_single.append({
                    'regime':       'single',
                    'concept':      concept,
                    'method':       method,
                    'layer':        layer_idx,
                    'label':        label,
                    'mmd2':         mmd,
                    'abs_skew_pre': sk_raw,
                    'abs_skew_post':sk_deb,
                    'abs_kurt_pre': ku_raw,
                    'abs_kurt_post':ku_deb,
                    'delta_skew':   sk_deb - sk_raw,
                    'delta_kurt':   ku_deb - ku_raw,
                })

df_single = pd.DataFrame(records_single)
print(f'Single: {len(df_single)} rows')

In [ ]:
records_seq = []

print('=== Sequential debiasing shift ===')
for concept in tqdm(CONCEPTS, desc='Concepts'):
    if concept not in df_split.columns:
        continue
    for method in METHODS:
        base = DEB_DIR / 'sequential' / concept / method / SPLIT
        if not base.exists():
            continue
        for layer_idx in range(NUM_LAYERS):
            pre_path  = base / f'layer_{layer_idx:02d}_pre.parquet'
            post_path = base / f'layer_{layer_idx:02d}_post.parquet'

            X_pre_pos, X_pre_neg = load_parquet_feats(pre_path,  concept, df_split)
            X_post_pos, X_post_neg = load_parquet_feats(post_path, concept, df_split)

            if X_pre_pos is None or X_post_pos is None:
                continue

            for label, X_pre, X_post in [
                ('pos', X_pre_pos, X_post_pos),
                ('neg', X_pre_neg, X_post_neg),
            ]:
                if len(X_pre) < 4 or len(X_post) < 4:
                    continue

                sk_pre,  ku_pre  = marginal_stats(X_pre)
                sk_post, ku_post = marginal_stats(X_post)
                mmd = mmd_rbf(X_pre, X_post)

                records_seq.append({
                    'regime':        'sequential',
                    'concept':       concept,
                    'method':        method,
                    'layer':         layer_idx,
                    'label':         label,
                    'mmd2':          mmd,
                    'abs_skew_pre':  sk_pre,
                    'abs_skew_post': sk_post,
                    'abs_kurt_pre':  ku_pre,
                    'abs_kurt_post': ku_post,
                    'delta_skew':    sk_post - sk_pre,
                    'delta_kurt':    ku_post - ku_pre,
                })

df_seq = pd.DataFrame(records_seq)
print(f'Sequential: {len(df_seq)} rows')

df_all = pd.concat([df_single, df_seq], ignore_index=True)
df_all.to_csv(OUT_DIR / 'results.csv', index=False)
print(f'Saved {len(df_all)} total rows')

## Visualization

In [ ]:
# MMD² per layer per method, averaged over concepts and pos/neg

for regime, df_r in [('single', df_single), ('sequential', df_seq)]:
    if df_r.empty:
        continue
    fig, ax = plt.subplots(figsize=(10, 4))
    for method in METHODS:
        sub = df_r[df_r['method'] == method].groupby('layer')['mmd2'].mean()
        if sub.empty:
            continue
        ax.plot(sub.index, sub.values, marker='o', ms=4, label=method)
    ax.set_xlabel('Layer')
    ax.set_ylabel('MMD² (avg over concepts & pos/neg)')
    ax.set_title(f'Distribution shift per debiasing step — {regime}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'mmd_{regime}.png', dpi=150)
    plt.show()

In [ ]:
# Heatmap: MMD² per concept × layer, for each method (single regime)

if not df_single.empty:
    for method in METHODS:
        sub = df_single[df_single['method'] == method]
        if sub.empty:
            continue
        pivot = sub.groupby(['concept', 'layer'])['mmd2'].mean().unstack('layer')

        fig, ax = plt.subplots(figsize=(14, 4))
        sns.heatmap(pivot, ax=ax, cmap='YlOrRd', annot=False, linewidths=0.3)
        ax.set_title(f'MMD² (raw vs debiased) — single, method={method}')
        ax.set_xlabel('Layer')
        plt.tight_layout()
        plt.savefig(OUT_DIR / f'heatmap_mmd_single_{method}.png', dpi=150)
        plt.show()

In [ ]:
# Delta kurtosis and skewness: does debiasing make distributions more or less normal?

for regime, df_r in [('single', df_single), ('sequential', df_seq)]:
    if df_r.empty:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, ylabel in [
        (axes[0], 'delta_kurt', '\u0394 |excess kurtosis| (post - pre)'),
        (axes[1], 'delta_skew', '\u0394 |skewness| (post - pre)'),
    ]:
        for method in METHODS:
            sub = df_r[df_r['method'] == method].groupby('layer')[metric].mean()
            if sub.empty:
                continue
            ax.plot(sub.index, sub.values, marker='o', ms=4, label=method)
        ax.axhline(0, color='black', lw=0.8, ls='--')
        ax.set_xlabel('Layer')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8)
    fig.suptitle(f'Change in distribution shape from debiasing — {regime}')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'delta_shape_{regime}.png', dpi=150)
    plt.show()